In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
import joblib

# ==========================================
# 1. ĐỌC DỮ LIỆU TỪ FILE CSV
# ==========================================
file_path = r"data_tranning\Car_sale_ads.csv"
print(f"Đang đọc dữ liệu từ: {file_path}")
df = pd.read_csv(file_path)

# Chỉ loại bỏ những dòng không có giá trị ở cột Price (vì đây là cái cần dự đoán)
df = df.dropna(subset=['Price'])

# ==========================================
# 2. KHAI BÁO CÁC TRƯỜNG DỮ LIỆU ĐẦU VÀO
# ==========================================
# Nhóm 1: Các trường dạng số học (Continuous/Numerical)
num_cols = [
    'Production_year', 'Mileage_km', 'Power_HP', 
    'Displacement_cm3', 'CO2_emissions', 'Doors_number'
]

# Nhóm 2: Các trường dạng chữ/phân loại (Categorical)
cat_cols = [
    'Condition', 'Vehicle_brand', 'Vehicle_model', 'Vehicle_version', 
    'Vehicle_generation', 'Fuel_type', 'Drive', 'Transmission', 
    'Type', 'Colour', 'Origin_country', 'First_owner'
]

X = df[num_cols + cat_cols]
y = df['Price']

# ==========================================
# 3. XÂY DỰNG BỘ TIỀN XỬ LÝ KÉP (PREPROCESSING)
# ==========================================
# Xử lý Nhóm 1: Nếu thiếu số -> Điền số trung bình -> Chuẩn hóa StandardScaler
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Xử lý Nhóm 2: Nếu thiếu chữ -> Điền chữ 'Missing' -> Mã hóa OneHotEncoder
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Gom 2 quy trình lại
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

# ==========================================
# 4. TẠO PIPELINE VÀ HUẤN LUYỆN
# ==========================================
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', KNeighborsRegressor(n_neighbors=5))
])

print("Bắt đầu huấn luyện mô hình (có thể mất nhiều thời gian do dữ liệu lớn)...")
model.fit(X, y)
print("Huấn luyện hoàn tất!")

# ==========================================
# 5. XUẤT RA FILE CHO BACKEND
# ==========================================
model_filename = 'ml_models/car_pricing_model.pkl'
joblib.dump(model, model_filename)
print(f"Đã lưu mô hình thành công vào file: {model_filename}")

Đang đọc dữ liệu từ: data_tranning\Car_sale_ads.csv
Bắt đầu huấn luyện mô hình (có thể mất nhiều thời gian do dữ liệu lớn)...
Huấn luyện hoàn tất!
Đã lưu mô hình thành công vào file: ml_models/car_pricing_model.pkl


In [1]:
import joblib
import pandas as pd

# ==========================================
# 1. TẢI MÔ HÌNH TỪ THƯ MỤC ML_MODELS
# ==========================================
model_path = 'ml_models/car_pricing_model.pkl'
print(f"Đang nạp mô hình AI từ: {model_path}...")

try:
    model = joblib.load(model_path)
    print("Nạp mô hình thành công!\n")
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file {model_path}. Hãy chắc chắn bạn đã chạy file training.")
    exit()

# ==========================================
# 2. TẠO DỮ LIỆU GIẢ LẬP (MÔ PHỎNG REQUEST TỪ FE)
# ==========================================
# Đây là thông tin của 1 chiếc xe cần định giá
# Lưu ý: Phải bọc giá trị bên trong list [] để tạo thành DataFrame hợp lệ
sample_data = {
    # Dữ liệu số
    # Dữ liệu số (Numeric)
    'Production_year': [2021],        # Năm sản xuất
    'Mileage_km': [30000],            # Số km đã đi (ODO)
    'Power_HP': [228],                # Công suất mã lực (Bản cao cấp 228 HP)
    'Displacement_cm3': [1998],       # Dung tích động cơ (2.0L)
    'CO2_emissions': [175],           # Lượng khí thải CO2 ước tính
    'Doors_number': [4],              # Số cửa (Sedan thường là 4 cửa)
    
    # Dữ liệu chữ / phân loại (Categorical)
    'Condition': ['Used'],            # Tình trạng: Đã qua sử dụng
    'Vehicle_brand': ['VinFast'],     # Hãng xe
    'Vehicle_model': ['Lux A2.0'],    # Dòng xe
    'Vehicle_version': ['Premium'],   # Phiên bản
    'Vehicle_generation': ['Gen 1'],  # Thế hệ
    'Fuel_type': ['Petrol'],          # Loại nhiên liệu: Xăng
    'Drive': ['RWD'],                 # Hệ dẫn động: Cầu sau (Rear-Wheel Drive)
    'Transmission': ['Automatic'],    # Hộp số: Tự động (ZF 8 cấp)
    'Type': ['Sedan'],                # Kiểu dáng: Sedan
    'Colour': ['Black'],              # Màu sắc
    'Origin_country': ['Vietnam'],    # Quốc gia sản xuất
    'First_owner': ['Yes']            # Chủ đầu tiên: Đúng
}

# Đưa dictionary vào Pandas DataFrame vì Pipeline của Scikit-learn yêu cầu
new_car_df = pd.DataFrame(sample_data)

# ==========================================
# 3. CHẠY DỰ ĐOÁN
# ==========================================
print("Đang tiến hành dự đoán giá...")
predicted_prices = model.predict(new_car_df)

# Lấy kết quả đầu tiên (vì ta chỉ truyền vào 1 chiếc xe)
final_price = predicted_prices[0]

# ==========================================
# 4. HIỂN THỊ KẾT QUẢ
# ==========================================
print("-" * 40)
print(f"THÔNG TIN XE:")
print(f"- Hãng: {sample_data['Vehicle_brand'][0]} {sample_data['Vehicle_model'][0]}")
print(f"- Đời: {sample_data['Production_year'][0]}")
print(f"- ODO: {sample_data['Mileage_km'][0]} km")
print("-" * 40)
print(f"💰 GIÁ DỰ ĐOÁN: {final_price:,.2f}")
print("-" * 40)

Đang nạp mô hình AI từ: ml_models/car_pricing_model.pkl...
Nạp mô hình thành công!

Đang tiến hành dự đoán giá...
----------------------------------------
THÔNG TIN XE:
- Hãng: VinFast Lux A2.0
- Đời: 2021
- ODO: 30000 km
----------------------------------------
💰 GIÁ DỰ ĐOÁN: 307,004.00
----------------------------------------
